# geo0641 fix 1 – Station ids as text in your S1 GeoPackage

**Why:** in geo0641 (session 1) we read the DWD station id as a number. The id is an **identifier**, not a quantity:
as a number, `00044` became `44` and the Zugspitze `05792` became `5792`. The DWD file names (`…_00044_…`) and later tables
(hourly precipitation in session 10, PostGIS) use the 5-character text form, so joins on the id would silently fail.
Rule of this course: **identifiers are text.**

**What this notebook does (2 minutes):** it opens your `data/derived/dwd_stations.gpkg` from session 1, turns `station_id` in every
layer into 5-character text (`44` → `00044`) and writes the layers back. Nothing else changes (geometry, CRS, other columns).

Run it once from top to bottom. It is safe to run it again.

In [ ]:
from pathlib import Path

import geopandas as gpd

GPKG = Path("data/derived/dwd_stations.gpkg")   # written by geo0641, section 6.1
assert GPKG.exists(), f"{GPKG} not found: run this notebook in your s01 folder (next to geo0641), or adjust the path"
gpd.list_layers(GPKG)

In [ ]:
for layer in gpd.list_layers(GPKG)["name"]:
    gdf = gpd.read_file(GPKG, layer=layer)
    if "station_id" not in gdf.columns:
        print(f"{layer}: no station_id column, left unchanged")
        continue
    before = gdf["station_id"].dtype
    gdf["station_id"] = gdf["station_id"].astype(str).str.zfill(5)
    gdf.to_file(GPKG, layer=layer, driver="GPKG")   # replaces this layer only, the other layers stay
    print(f"{layer}: {len(gdf)} stations, station_id {before} -> text, e.g. {gdf['station_id'].iloc[0]}")

In [ ]:
# Check: read the layers again
for layer in gpd.list_layers(GPKG)["name"]:
    gdf = gpd.read_file(GPKG, layer=layer)
    if "station_id" in gdf.columns:
        assert (gdf["station_id"].str.len() == 5).all(), layer
        print(f"ok  {layer}: {gdf['station_id'].dtype}, {gdf['station_id'].head(3).tolist()}, CRS EPSG:{gdf.crs.to_epsg()}")

## QGIS
If the GeoPackage is open in a QGIS project, download it again from the JupyterHub, then remove the layers and add them again
(or *Layer Properties → Source* → reload). *Layer Properties → Fields* now shows `station_id` as **Text (string)**, and the labels
read `00044 – Name`.

## For the future
If you run geo0641 again, add the id to the `dtype` argument in section 3.2:

```python
dtype={"station_id": str, "date_from": str, "date_to": str}
```